# Angrist & Lavy (1999) — Data Exploration
Quick look at the 4th grade dataset before replication.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --- Load data ---
# Update path to wherever your file lives
df = pd.read_stata('final4.dta')

# Compute Maimonides Rule predicted class size
df['fsc'] = df['cohsize'] / (np.floor((df['cohsize'] - 1) / 40) + 1)

# Restrict to enrollment <= 220 (matches paper's N=2049)
df = df[df['cohsize'] <= 220].copy()

print('Rows:', len(df))
print('Schools:', df['schlcode'].nunique())

Rows: 2053
Schools: 1015


In [ ]:
df

## 1. Summary statistics for key variables

In [ ]:
key_vars = ['cohsize', 'classize', 'fsc', 'avgverb', 'avgmath', 'tipuach']
labels   = ['Enrollment', 'Actual class size', 'Maimonides predicted size',
            'Avg reading score', 'Avg math score', '% disadvantaged']

summary = df[key_vars].describe().T[['mean','std','min','max']].round(2)
summary.index = labels
summary

## 2. Distribution of enrollment (the running variable)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(df['cohsize'], bins=50, color='steelblue', edgecolor='white')

# Mark the Maimonides cutoffs at 40, 80, 120, 160, 200
for cut in [40, 80, 120, 160, 200]:
    ax.axvline(cut, color='red', linestyle='--', linewidth=1, label='Cutoff' if cut == 40 else '')

ax.set_xlabel('Enrollment count')
ax.set_ylabel('Number of classes')
ax.set_title('Distribution of Enrollment with Maimonides Cutoffs')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Actual vs. Maimonides predicted class size (Figure 1 Panel B preview)
Average actual class size and Maimonides Rule by enrollment count.

In [ ]:
# Average actual class size and fsc by each enrollment value
by_enroll = df.groupby('cohsize').agg(
    actual=('classize', 'mean'),
    maimonides=('fsc', 'mean')
).reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(by_enroll['cohsize'], by_enroll['maimonides'], 'k--', linewidth=1.5, label='Maimonides Rule')
ax.plot(by_enroll['cohsize'], by_enroll['actual'],     'k-',  linewidth=1.5, label='Actual class size')

for cut in [40, 80, 120, 160, 200]:
    ax.axvline(cut, color='gray', linestyle=':', linewidth=0.8)

ax.set_xlabel('Enrollment count')
ax.set_ylabel('Class size')
ax.set_title('Figure 1 Panel B — Fourth Grade')
ax.legend()
ax.set_ylim(5, 45)
plt.tight_layout()
plt.show()

## 4. Test scores vs. actual class size (naive OLS picture)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, yvar, title in zip(axes,
                           ['avgverb', 'avgmath'],
                           ['Reading comprehension', 'Math']):
    sub = df.dropna(subset=[yvar])
    ax.scatter(sub['classize'], sub[yvar], alpha=0.15, s=10, color='steelblue')

    # OLS fit line
    m = np.polyfit(sub['classize'], sub[yvar], 1)
    xs = np.linspace(sub['classize'].min(), sub['classize'].max(), 100)
    ax.plot(xs, np.polyval(m, xs), 'r-', linewidth=2,
            label=f'OLS slope = {m[0]:.3f}')

    ax.set_xlabel('Actual class size')
    ax.set_ylabel('Average score')
    ax.set_title(title)
    ax.legend(fontsize=9)

fig.suptitle('Naive OLS: Test scores vs. class size (4th grade)', fontsize=12)
plt.tight_layout()
plt.show()

## 5. Test scores vs. Maimonides predicted class size (reduced-form RDD picture)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, yvar, title in zip(axes,
                           ['avgverb', 'avgmath'],
                           ['Reading comprehension', 'Math']):
    by_fsc = df.dropna(subset=[yvar]).groupby('cohsize').agg(
        fsc=(  'fsc',  'mean'),
        score=(yvar,   'mean')
    ).reset_index()

    ax.scatter(by_fsc['fsc'], by_fsc['score'], alpha=0.4, s=12, color='steelblue')

    m = np.polyfit(by_fsc['fsc'], by_fsc['score'], 1)
    xs = np.linspace(by_fsc['fsc'].min(), by_fsc['fsc'].max(), 100)
    ax.plot(xs, np.polyval(m, xs), 'r-', linewidth=2,
            label=f'slope = {m[0]:.3f}')

    ax.set_xlabel('Maimonides predicted class size')
    ax.set_ylabel('Average score')
    ax.set_title(title)
    ax.legend(fontsize=9)

fig.suptitle('Reduced-form: Test scores vs. Maimonides predicted class size', fontsize=12)
plt.tight_layout()
plt.show()

## 6. Quick OLS numbers (Table II cols 7 & 10 preview)

In [ ]:
def ols_clustered(y, X, clusters):
    """OLS with cluster-robust standard errors."""
    XtX_inv = np.linalg.inv(X.T @ X)
    beta  = XtX_inv @ X.T @ y
    resid = y - X @ beta
    n, k  = X.shape
    G     = len(np.unique(clusters))
    meat  = np.zeros((k, k))
    for g in np.unique(clusters):
        idx = clusters == g
        meat += X[idx].T @ np.outer(resid[idx], resid[idx]) @ X[idx]
    correction = (G / (G - 1)) * ((n - 1) / (n - k))
    se   = np.sqrt(np.diag(XtX_inv @ meat @ XtX_inv * correction))
    rmse = np.sqrt((resid @ resid) / (n - k))
    r2   = 1 - (resid @ resid) / ((y - y.mean()) ** 2).sum()
    return beta, se, rmse, r2, n

results = []
for yvar, label in [('avgverb', 'Reading (col 7)'), ('avgmath', 'Math (col 10)')]:
    sub = df.dropna(subset=[yvar, 'classize'])
    y   = sub[yvar].values
    X   = np.column_stack([np.ones(len(sub)), sub['classize'].values])
    beta, se, rmse, r2, n = ols_clustered(y, X, sub['schlcode'].values)
    results.append({
        'Outcome': label,
        'Class size coef': round(beta[1], 3),
        'Std error':       round(se[1],   3),
        'RMSE':            round(rmse,     2),
        'R²':              round(r2,       3),
        'N':               int(n)
    })

pd.DataFrame(results).set_index('Outcome')

## 7. Quick reduced-form numbers (Table III Panel A cols 7, 9, 11 preview)

In [ ]:
results_rf = []
specs = [
    ('classize', 'Class size (col 7)'),
    ('avgverb',  'Reading (col 9)'),
    ('avgmath',  'Math (col 11)'),
]

for yvar, label in specs:
    sub = df.dropna(subset=[yvar, 'fsc', 'tipuach'])
    y   = sub[yvar].values
    X   = np.column_stack([np.ones(len(sub)), sub['fsc'].values, sub['tipuach'].values])
    beta, se, rmse, r2, n = ols_clustered(y, X, sub['schlcode'].values)
    results_rf.append({
        'Outcome':   label,
        'fsc coef':  round(beta[1], 3),
        'Std error': round(se[1],   3),
        'RMSE':      round(rmse,    2),
        'R²':        round(r2,      3),
        'N':         int(n)
    })

pd.DataFrame(results_rf).set_index('Outcome')